# UNIVERSIDAD NACIONAL DE SAN CRISTÓBAL DE HUAMANGA
## Escuela Profesional de Ingeniería de Sistemas
### INTELIGENCIA ARTIFICIAL I (IS-484) - 2026-II

**Informe de Laboratorio N° 01: Fundamentos de IA y Agentes Inteligentes**

* **Docente:** Ing. Leidy Rosmery Maldonado Chauca
* **Estudiante:** Alexander Gómez de la Cruz
* **Fecha de Entrega:** 14 de Septiembre de 2026

---

### Objetivos de la Práctica
1. Clasificar sistemas de Inteligencia Artificial según el tipo de problema y área de aplicación.
2. Representar un sistema mediante el esquema formal de agente inteligente (PEAS), determinando su racionalidad y si corresponde a IA estrecha o AGI.
3. Configurar y verificar el entorno virtual de trabajo en Python mediante Jupyter Notebook con sus bibliotecas oficiales.
4. Diseñar e implementar agentes reactivos simples y de múltiples percepciones en Python.

## 1. Verificación del Entorno y Bibliotecas
Se realiza la importación de los paquetes principales de Python y se comprueban las versiones instaladas en el entorno virtual de trabajo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import sklearn
import scipy

print("==================================================")
print(" VERIFICACIÓN DE ENTORNO Y BIBLIOTECAS (IS-484)")
print("==================================================")
print(f"NumPy:        {np.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"Matplotlib:   {matplotlib.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"SciPy:        {scipy.__version__}")
print("==================================================")

## 2. Simulación y Visualización de Transacciones Bancarias
Generación de un conjunto de datos sintéticos de 300 transacciones analizando la probabilidad de fraude en función de la hora y el monto consumido.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
n = 300

hora = np.random.randint(0, 24, n)
monto = np.random.exponential(scale=150, size=n) + 10

prob_fraude = 0.03 + 0.35 * ((hora <= 5) | (hora >= 23)) + 0.25 * (monto > 400)
prob_fraude = np.clip(prob_fraude, 0, 0.9)
es_fraude = (np.random.rand(n) < prob_fraude).astype(int)

transacciones = pd.DataFrame({"hora": hora, "monto": monto, "es_fraude": es_fraude})

colores = {0: "#0ca30c", 1: "#d03b3b"}
etiquetas = {0: "Legítima", 1: "Fraude"}

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

for valor in [0, 1]:
    subset = transacciones[transacciones["es_fraude"] == valor]
    ax.scatter(
        subset["hora"], subset["monto"],
        s=70, alpha=0.75,
        c=colores[valor],
        edgecolors="white", linewidths=0.6,
        label=etiquetas[valor],
    )

ax.set_title("Transacciones por hora y monto", fontsize=15, color="#0b0b0b", pad=14)
ax.set_xlabel("Hora del dia", fontsize=11, color="#52514e")
ax.set_ylabel("Monto (S/)", fontsize=11, color="#52514e")
ax.set_xticks(range(0, 24, 2))
ax.grid(True, color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Tipo de transaccion", frameon=False, loc="upper right")

plt.tight_layout()
plt.show()

print("Total transacciones:", len(transacciones), "| Fraudes:", transacciones["es_fraude"].sum())

## 3. Clasificación de Sistemas e Identificación PEAS

### 3.1 Clasificación de Sistemas de IA

| Sistema Elegido | Entrada (Input) | Salida (Output) | Tipo de Problema | Área Predominante de IA |
|---|---|---|---|---|
| **1. Filtro Anti-Spam de Gmail** | Texto del correo, dirección del remitente, IP y enlaces. | Clasificación: *Bandeja de Entrada* o *Spam*. | Clasificación Binaria | Procesamiento de Lenguaje Natural (NLP) |
| **2. Recomendador de Netflix** | Historial de reproducciones, valoraciones del usuario, horarios de uso. | Lista ordenada de títulos sugeridos con porcentaje de afinidad. | Ranking / Filtrado Colaborativo | Sistemas de Recomendación |

---

### 3.2 Ficha PEAS: Sistema Anti-Fraude Bancario en Tiempo Real

* **Medida de Desempeño (P):** Maximización de fraudes detectados, minimización de falsos positivos y reducción de pérdidas financieras netas.
* **Entorno (E):** Servidores bancarios, red de pagos en tiempo real, pasarelas e-commerce, cajeros automáticos y usuarios.
* **Actuadores (A):** Aprobar transacción, rechazar/bloquear operación, solicitar autenticación 2FA, emitir alerta de seguridad.
* **Sensores (S):** API de la pasarela de pagos, geolocalización IP/GPS, registros de sesión del cliente, token del dispositivo.

#### Análisis del Agente
* **Tipo de Agente:** Agente de Aprendizaje / Basado en Modelos.
* **Racionalidad:** **Sí es racional**, ya que ejecuta la acción que maximiza su medida de desempeño esperada en base a sus percepciones y modelo.
* **Clasificación de IA:** **IA Estrecha (Narrow AI)**, optimizada exclusivamente para detectar anomalías en transacciones financieras.

## 4. Agente Reactivo Simple: Control de Termostato

### 4.1 Ficha PEAS
| Elemento | Descripción |
|---|---|
| **Percepción (S)** | Lectura instantánea de la temperatura actual en °C mediante el sensor. |
| **Acciones (A)** | Encender Aire Acondicionado, Encender Calefacción, Apagar Climatizador. |
| **Entorno (E)** | Recinto o habitación cerrada. |
| **Objetivo** | Mantener el ambiente dentro de un rango confortable (22 °C ± 1 °C). |
| **Medida de Desempeño (P)** | Porcentaje de tiempo en rango confortable y ahorro de energía. |

### 4.2 Reglas de Decisión
| Condición Térmica | Acción a tomar |
|---|---|
| `diferencia > margen` (Muy por encima de la deseada) | Encender Aire Acondicionado |
| `diferencia < -margen` (Muy por debajo de la deseada) | Encender Calefacción |
| `-margen <= diferencia <= margen` (Cerca de la deseada) | Apagar Climatizador |

In [ ]:
def agente_termostato(temperatura_actual, temperatura_objetivo=22.0, margen=1.0):
    """Agente reactivo simple para el control de temperatura."""
    diferencia = temperatura_actual - temperatura_objetivo
    
    if diferencia > margen:
        return "Encender Aire Acondicionado"
    elif diferencia < -margen:
        return "Encender Calefacción"
    else:
        return "Apagar Climatizador"


# Simulación con secuencia de percepciones térmicas
percepciones = [18.0, 19.5, 21.8, 24.3, 26.0, 23.0, 20.5, 21.0, 23.0]

print("--- SIMULACIÓN AGENTE TERMOSTATO ---")
for t in percepciones:
    accion = agente_termostato(t)
    print(f"Percepcion: {t:4.1f} °C -> Accion del agente: {accion}")

### 4.3 Reflexión sobre el Agente Termostato

1. **¿Es racional este agente?**  
   **Sí es racional.** Dentro del marco de un agente reactivo simple, elige la acción que maximiza la medida de desempeño esperada en base a la percepción del instante.

2. **¿Bajo qué condiciones del entorno dejaría de serlo?**  
   * **Falla en sensores:** Si el sensor reporta lecturas erróneas (ej. marca 30°C cuando la real es 15°C).
   * **Margen muy amplio:** Si la tolerancia fuera de ±5°C, causaría oscilaciones térmicas incómodas.
   * **Entorno abierto:** Si las ventanas están abiertas, el climatizador gastará energía indefinidamente sin estabilizar la temperatura.

## 5. Agente de Múltiples Percepciones: Riego Automático

### 5.1 Ficha PEAS
| Elemento | Descripción |
|---|---|
| **Percepciones (S)** | Porcentaje de humedad del suelo (%) y pronóstico de lluvia (Booleano). |
| **Acciones (A)** | Riego Intensivo, Riego Moderado, No Regar. |
| **Entorno (E)** | Jardín o terreno agrícola exterior. |
| **Objetivo** | Mantener la humedad adecuada del suelo optimizando el consumo de agua. |
| **Medida de Desempeño (P)** | Salud de la vegetación y litros de agua ahorrados aprovechando las lluvias. |

### 5.2 Reglas de Decisión Combinadas
* **Suelo seco (< 30%) Y NO lloverá:** `Regar - Riego Intensivo`
* **Suelo seco (< 30%) Y SÍ lloverá:** `No Regar (Esperar Lluvia Pronosticada)` *(Conflicto resuelto)*
* **Humedad media (30% - 60%) Y NO lloverá:** `Regar - Riego Moderado`
* **Otros casos (Suelo húmedo o lluvia inminente):** `No Regar (Suelo Adecuado)`